<a href="https://colab.research.google.com/github/i2mmmmm/train_project/blob/main/20230820_transformer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import LayerNormalization, MultiHeadAttention, Flatten
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


In [2]:
from google.colab import drive

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
s30_train = pd.read_csv('/content/drive/My Drive/철도/s30_train.csv')
s40_train = pd.read_csv('/content/drive/My Drive/철도/s40_train.csv')
s50_train = pd.read_csv('/content/drive/My Drive/철도/s50_train.csv')
s70_train = pd.read_csv('/content/drive/My Drive/철도/s70_train.csv')
s100_train = pd.read_csv('/content/drive/My Drive/철도/s100_train.csv')
c30_train = pd.read_csv('/content/drive/My Drive/철도/c30_train.csv')
c40_train = pd.read_csv('/content/drive/My Drive/철도/c40_train.csv')
c50_train = pd.read_csv('/content/drive/My Drive/철도/c50_train.csv')
c70_train = pd.read_csv('/content/drive/My Drive/철도/c70_train.csv')
c100_train = pd.read_csv('/content/drive/My Drive/철도/c100_train.csv')

In [35]:
def prepare_data(filename):
    data = filename
    X = data.drop(columns=["YL_M1_B1_W1", "YR_M1_B1_W1", "YL_M1_B1_W2", "YR_M1_B1_W2"])
    y = data[["YL_M1_B1_W1", "YR_M1_B1_W1", "YL_M1_B1_W2", "YR_M1_B1_W2"]]

    # 데이터 스케일링
    scaler = StandardScaler()
    X = scaler.fit_transform(X)

    return X, y
def reshape_data(X, y):
    X_list, y_list = [], []
    X_padded = np.vstack([np.zeros((N_TIMESTEPS, X.shape[1])), X])

    for i in range(len(X_padded) - N_TIMESTEPS):
        X_list.append(X_padded[i:i+N_TIMESTEPS])
        y_list.append(y.iloc[i])

    return np.array(X_list), np.array(y_list)

data_s_X, data_s_y = prepare_data(c100_train)

N_TIMESTEPS = 10
N_FEATURES = 34

def transformer_encoder(inputs):
    # Self attention
    attention = MultiHeadAttention(num_heads=8, key_dim=N_FEATURES)(inputs, inputs)
    attention = Dropout(0.3)(attention)
    add_attention = tf.keras.layers.Add()([inputs, attention])
    attention_norm = LayerNormalization(epsilon=1e-6)(add_attention)

    # Feed forward
    ff = Dense(units=512, activation="relu")(attention_norm)
    ff = Dense(units=N_FEATURES)(ff)
    ff = Dropout(0.3)(ff)
    add_ff = tf.keras.layers.Add()([attention_norm, ff])
    return LayerNormalization(epsilon=1e-6)(add_ff)



In [36]:
# 포지셔널 인코딩 함수
def positional_encoding(position, d_model):
    def get_angles(pos, i, d_model):
        angle_rates = 1 / np.power(10000, (2 * (i // 2)) / np.float32(d_model))
        return pos * angle_rates

    angle_rads = get_angles(np.arange(position)[:, np.newaxis],
                            np.arange(d_model)[np.newaxis, :],
                            d_model)

    # apply sin to even indices in the array; 2i
    angle_rads[:, 0::2] = np.sin(angle_rads[:, 0::2])

    # apply cos to odd indices in the array; 2i+1
    angle_rads[:, 1::2] = np.cos(angle_rads[:, 1::2])

    pos_encoding = angle_rads[np.newaxis, ...]

    return tf.cast(pos_encoding, dtype=tf.float32)

def transformer_block(inputs):
    # Self attention
    attention = MultiHeadAttention(num_heads=8, key_dim=N_FEATURES)(inputs, inputs)
    attention = Dropout(0.5)(attention)
    add_attention = tf.keras.layers.Add()([inputs, attention])
    attention_norm = LayerNormalization(epsilon=1e-6)(add_attention)

    # Feed forward
    ff = Dense(units=512, activation="relu")(attention_norm)
    ff = Dense(units=N_FEATURES)(ff)
    ff = Dropout(0.5)(ff)
    add_ff = tf.keras.layers.Add()([attention_norm, ff])

    return LayerNormalization(epsilon=1e-6)(add_ff)

# 모델 구축
inputs = tf.keras.Input(shape=(N_TIMESTEPS, N_FEATURES))

# 포지셔널 인코딩 추가
x = inputs + positional_encoding(N_TIMESTEPS, N_FEATURES)

# 3개의 transformer 블록을 추가
for _ in range(3):
    x = transformer_block(x)

x = Flatten()(x)
outputs = Dense(4)(x)

model = tf.keras.Model(inputs=inputs, outputs=outputs)

model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), loss='mse', metrics=['mae'])

X_s, y_s = reshape_data(data_s_X, data_s_y)

# 트레이닝/테스트 데이터 나누기
X_train_s = X_s[:10001]
y_train_s = y_s[:10001]
X_test_s = X_s[10001:]
y_test_s = y_s[10001:]

# Early stopping
es = EarlyStopping(monitor='val_loss', mode='min', verbose=1, patience=20)

# 모델 학습
model.fit(X_train_s, y_train_s, epochs=50, batch_size=64, validation_data=(X_test_s, y_test_s), callbacks=[es])

# 예측
predictions_s = model.predict(X_test_s)

# 예측 결과 출력
print("Predictions for data_s")
print(predictions_s[:5])


Epoch 1/50
157/157 [==============================] - 22s 98ms/step - loss: 0.2102 - mae: 0.3300 - val_loss: 0.0035 - val_mae: 0.0469
Epoch 2/50
157/157 [==============================] - 14s 88ms/step - loss: 0.0258 - mae: 0.1248 - val_loss: 0.0032 - val_mae: 0.0488
Epoch 3/50
157/157 [==============================] - 15s 96ms/step - loss: 0.0098 - mae: 0.0771 - val_loss: 4.8386e-04 - val_mae: 0.0172
Epoch 4/50
157/157 [==============================] - 15s 94ms/step - loss: 0.0055 - mae: 0.0574 - val_loss: 2.8222e-04 - val_mae: 0.0146
Epoch 5/50
157/157 [==============================] - 14s 93ms/step - loss: 0.0038 - mae: 0.0476 - val_loss: 7.9928e-04 - val_mae: 0.0227
Epoch 6/50
157/157 [==============================] - 14s 91ms/step - loss: 0.0031 - mae: 0.0424 - val_loss: 2.9212e-04 - val_mae: 0.0142
Epoch 7/50
157/157 [==============================] - 14s 92ms/step - loss: 0.0025 - mae: 0.0382 - val_loss: 3.9781e-04 - val_mae: 0.0170
Epoch 8/50
157/157 [======================

In [7]:
predictions_30s = pd.DataFrame(predictions_s, columns=['YL_M1_B1_W1', 'YR_M1_B1_W1', 'YL_M1_B1_W2', 'YR_M1_B1_W2'])

In [10]:
predictions_40s = pd.DataFrame(predictions_s, columns=['YL_M1_B1_W1', 'YR_M1_B1_W1', 'YL_M1_B1_W2', 'YR_M1_B1_W2'])

In [13]:
predictions_50s = pd.DataFrame(predictions_s, columns=['YL_M1_B1_W1', 'YR_M1_B1_W1', 'YL_M1_B1_W2', 'YR_M1_B1_W2'])

In [16]:
predictions_70s = pd.DataFrame(predictions_s, columns=['YL_M1_B1_W1', 'YR_M1_B1_W1', 'YL_M1_B1_W2', 'YR_M1_B1_W2'])

In [19]:
predictions_100s = pd.DataFrame(predictions_s, columns=['YL_M1_B1_W1', 'YR_M1_B1_W1', 'YL_M1_B1_W2', 'YR_M1_B1_W2'])

In [23]:
predictions_30c = pd.DataFrame(predictions_s, columns=['YL_M1_B1_W1', 'YR_M1_B1_W1', 'YL_M1_B1_W2', 'YR_M1_B1_W2'])

In [27]:
predictions_40c = pd.DataFrame(predictions_s, columns=['YL_M1_B1_W1', 'YR_M1_B1_W1', 'YL_M1_B1_W2', 'YR_M1_B1_W2'])

In [30]:
predictions_50c = pd.DataFrame(predictions_s, columns=['YL_M1_B1_W1', 'YR_M1_B1_W1', 'YL_M1_B1_W2', 'YR_M1_B1_W2'])

In [34]:
predictions_70c = pd.DataFrame(predictions_s, columns=['YL_M1_B1_W1', 'YR_M1_B1_W1', 'YL_M1_B1_W2', 'YR_M1_B1_W2'])

In [37]:
predictions_100c = pd.DataFrame(predictions_s, columns=['YL_M1_B1_W1', 'YR_M1_B1_W1', 'YL_M1_B1_W2', 'YR_M1_B1_W2'])

In [38]:
answer_sample = pd.read_csv('/content/drive/My Drive/철도/answer_sample.csv')
c30 = predictions_30c
c40 = predictions_40c
c50 = predictions_50c
c70 = predictions_70c
c100 = predictions_100c
s30 = predictions_30s
s40 = predictions_40s
s50 = predictions_50s
s70 = predictions_70s
s100 = predictions_100s

In [39]:
c30.columns = ['YL_M1_B1_W1_c30', 'YR_M1_B1_W1_c30', 'YL_M1_B1_W2_c30', 'YR_M1_B1_W2_c30']
c40.columns = ['YL_M1_B1_W1_c40', 'YR_M1_B1_W1_c40', 'YL_M1_B1_W2_c40', 'YR_M1_B1_W2_c40']
c50.columns = ['YL_M1_B1_W1_c50', 'YR_M1_B1_W1_c50', 'YL_M1_B1_W2_c50', 'YR_M1_B1_W2_c50']
c70.columns = ['YL_M1_B1_W1_c70', 'YR_M1_B1_W1_c70', 'YL_M1_B1_W2_c70', 'YR_M1_B1_W2_c70']
c100.columns = ['YL_M1_B1_W1_c100', 'YR_M1_B1_W1_c100', 'YL_M1_B1_W2_c100', 'YR_M1_B1_W2_c100']

s30.columns = ['YL_M1_B1_W1_s30', 'YR_M1_B1_W1_s30', 'YL_M1_B1_W2_s30', 'YR_M1_B1_W2_s30']
s40.columns = ['YL_M1_B1_W1_s40', 'YR_M1_B1_W1_s40', 'YL_M1_B1_W2_s40', 'YR_M1_B1_W2_s40']
s50.columns = ['YL_M1_B1_W1_s50', 'YR_M1_B1_W1_s50', 'YL_M1_B1_W2_s50', 'YR_M1_B1_W2_s50']
s70.columns = ['YL_M1_B1_W1_s70', 'YR_M1_B1_W1_s70', 'YL_M1_B1_W2_s70', 'YR_M1_B1_W2_s70']
s100.columns = ['YL_M1_B1_W1_s100', 'YR_M1_B1_W1_s100', 'YL_M1_B1_W2_s100', 'YR_M1_B1_W2_s100']

answer = pd.concat([answer_sample.Distance, s30,s40,s50,s70,s100,c30,c40,c50,c70,c100], axis=1)

answer

,Distance,YL_M1_B1_W1_s30,YR_M1_B1_W1_s30,YL_M1_B1_W2_s30,YR_M1_B1_W2_s30,YL_M1_B1_W1_s40,YR_M1_B1_W1_s40,YL_M1_B1_W2_s40,YR_M1_B1_W2_s40,YL_M1_B1_W1_s50,...,YL_M1_B1_W2_c50,YR_M1_B1_W2_c50,YL_M1_B1_W1_c70,YR_M1_B1_W1_c70,YL_M1_B1_W2_c70,YR_M1_B1_W2_c70,YL_M1_B1_W1_c100,YR_M1_B1_W1_c100,YL_M1_B1_W2_c100,YR_M1_B1_W2_c100
0,2500.25,0.033074,0.005443,0.055163,-0.060068,-0.013514,0.007482,-0.006955,0.022198,-0.007451,...,0.014765,-0.003969,0.024467,-0.005222,-0.013589,0.004508,-0.002887,0.003601,-0.000087,0.016028
1,2500.50,0.033612,0.006224,0.062348,-0.066848,-0.013686,0.007527,-0.003724,0.018554,-0.006704,...,0.023504,-0.013630,0.022659,-0.003699,-0.002659,-0.004279,-0.002502,0.004838,0.007232,0.007887
2,2500.75,0.034559,0.007373,0.072114,-0.075882,-0.014684,0.007819,0.001576,0.013974,-0.008193,...,0.034604,-0.024017,0.018619,0.000333,0.011238,-0.016879,-0.003634,0.008746,0.016217,-0.002642
3,2501.00,0.033477,0.009194,0.078061,-0.079411,-0.015517,0.008758,0.005792,0.009528,-0.011156,...,0.041899,-0.030179,0.013423,0.003616,0.024656,-0.028998,-0.007932,0.011679,0.022454,-0.011045
4,2501.25,0.031178,0.011391,0.077163,-0.079074,-0.015930,0.009702,0.008201,0.006845,-0.013492,...,0.046052,-0.033582,0.011123,0.006904,0.035984,-0.039342,-0.010096,0.012611,0.028425,-0.015447
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1994,2998.75,0.025585,0.012201,0.052026,-0.056692,-0.008143,0.003508,0.019643,-0.002245,0.004531,...,0.041141,-0.031533,0.032705,-0.014134,0.034595,-0.043872,0.004952,-0.003007,0.032110,-0.017677
1995,2999.00,0.025004,0.012005,0.042805,-0.049633,-0.007394,0.002764,0.015967,0.001177,0.003410,...,0.036500,-0.027354,0.031505,-0.013990,0.028977,-0.037016,0.005567,-0.001846,0.027003,-0.011916
1996,2999.25,0.024296,0.011213,0.031481,-0.041452,-0.007998,0.003309,0.011187,0.005109,0.001016,...,0.032928,-0.024026,0.027782,-0.011279,0.019190,-0.026943,0.002823,0.000785,0.019387,-0.004990
1997,2999.50,0.022656,0.010723,0.021027,-0.032331,-0.009135,0.003811,0.007075,0.009090,-0.000566,...,0.027575,-0.017877,0.026319,-0.009236,0.008393,-0.016880,-0.001564,0.002292,0.011377,0.001943


In [40]:
answer.to_csv('/content/drive/My Drive/철도/answer20230820_tf.csv', index=False)

In [8]:


# 예측과 실제값 간의 MAE, MSE, RMSE, R-squared 계산
mae = mean_absolute_error(y_test_s, predictions_s)
mse = mean_squared_error(y_test_s, predictions_s)
rmse = np.sqrt(mse)
r2 = r2_score(y_test_s, predictions_s)

# 결과 출력
print("Evaluation metrics for data_s:")
print("MAE:", mae)
print("MSE:", mse)
print("RMSE:", rmse)
print("R-squared:", r2)


Evaluation metrics for data_s:
MAE: 0.028660301391699096
MSE: 0.0019187690603298217
RMSE: 0.04380375623539404
R-squared: 0.2635515558019301


In [ ]:
# 예측 결과 DataFrame 생성
predictions_df = pd.DataFrame(predictions_s, columns=["Pred_YL_M1_B1_W1", "Pred_YR_M1_B1_W1", "Pred_YL_M1_B1_W2", "Pred_YR_M1_B1_W2"])

# CSV로 저장
predictions_df.to_csv("/Users/leeyoungdong/predictions100.csv", index=False)